In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
# ================================
# 🔥 ABLATION STUDY - FULL PIPELINE
# ================================

import numpy as np

# ----------------------------------
# 🔹 CONFIG
# ----------------------------------

class AblationConfig:
    def __init__(self, use_social=True, use_satellite=True, use_gcn=True):
        self.use_social = use_social
        self.use_satellite = use_satellite
        self.use_gcn = use_gcn


# ----------------------------------
# 🔹 RISK SCORE COMPUTATION
# ----------------------------------

def compute_risk_score(zone, config):
    """
    zone dict must contain:
    - satellite_damage
    - social_severity
    - urgency_score
    """

    # Satellite signal
    sat_score = zone.get("satellite_damage", 0) if config.use_satellite else 0

    # Social signal
    social_score = 0
    if config.use_social:
        social_score = (
            0.6 * zone.get("social_severity", 0) +
            0.4 * zone.get("urgency_score", 0)
        )

    # Combined score
    base_score = 0.7 * sat_score + 0.3 * social_score

    return base_score


# ----------------------------------
# 🔹 GCN REFINEMENT
# ----------------------------------

def apply_gcn_refinement(zones, scores, config):

    if not config.use_gcn:
        return scores

    refined = {}

    for z in zones:
        zid = z["id"]
        neighbors = z.get("neighbors", [])

        if neighbors:
            neighbor_vals = [scores[n] for n in neighbors if n in scores]
            spatial_avg = np.mean(neighbor_vals) if neighbor_vals else 0
            refined[zid] = 0.7 * scores[zid] + 0.3 * spatial_avg
        else:
            refined[zid] = scores[zid]

    return refined


# ----------------------------------
# 🔹 BUILD RISK MAP
# ----------------------------------

def build_zone_risk(zones, config):

    scores = {}
    for z in zones:
        scores[z["id"]] = compute_risk_score(z, config)

    scores = apply_gcn_refinement(zones, scores, config)

    return scores


# ----------------------------------
# 🔹 PPO EPISODE RUNNER
# ----------------------------------

def run_ppo_episode(env, model):
    obs = env.reset()
    done = False
    total_saved = 0

    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, info = env.step(action)

        # IMPORTANT: your env must return this
        total_saved += info.get("people_saved", 0)

    return total_saved


# ----------------------------------
# 🔹 ABLATION EXECUTION
# ----------------------------------

def run_ablation(zones, env_builder, model):

    variants = {
        "Full PPO": AblationConfig(True, True, True),
        "w/o Social Media": AblationConfig(False, True, True),
        "w/o Satellite Images": AblationConfig(True, False, True),
        "w/o GCN": AblationConfig(True, True, False),
        "All Removed": AblationConfig(False, False, False),
    }

    results = []
    total_people = sum(z["population"] for z in zones)

    for name, config in variants.items():

        # 🔥 recompute risk scores
        risk_scores = build_zone_risk(zones, config)

        # 🔥 rebuild environment using new risk
        env = env_builder(zones, risk_scores)

        # 🔥 run PPO policy
        saved = run_ppo_episode(env, model)

        save_rate = (saved / total_people) * 100

        results.append({
            "variant": name,
            "people_saved": int(saved),
            "save_rate": round(save_rate, 2)
        })

    return results


# ----------------------------------
# 🔹 DELTA COMPUTATION
# ----------------------------------

def compute_delta(results):

    full_rate = next(r for r in results if r["variant"] == "Full PPO")["save_rate"]

    for r in results:
        if r["variant"] == "Full PPO":
            r["delta"] = "-"
        else:
            r["delta"] = round(r["save_rate"] - full_rate, 2)

    return results


# ----------------------------------
# 🔹 FINAL FUNCTION (CALL THIS)
# ----------------------------------

def run_full_ablation(zones, env_builder, model):
    results = run_ablation(zones, env_builder, model)
    results = compute_delta(results)
    return results


# ================================
# ✅ USAGE
# ================================

# results = run_full_ablation(zones, env_builder, model)
# print(results)


📊 ABLATION STUDY — FRAMEWORK COMPONENT CONTRIBUTION
╒══════════════════════╤════════════════╤═════════════════════════╤═════════════════╕
│ Variant              │ People Saved   │ Overall Save Rate (%)   │ Δ vs Full PPO   │
╞══════════════════════╪════════════════╪═════════════════════════╪═════════════════╡
│ Full PPO             │ 45,866         │ 92.1%                   │ -               │
├──────────────────────┼────────────────┼─────────────────────────┼─────────────────┤
│ w/o Social Media     │ 43,250         │ 86.8%                   │ -5.3 pp         │
├──────────────────────┼────────────────┼─────────────────────────┼─────────────────┤
│ w/o Satellite Images │ 40,460         │ 81.2%                   │ -10.9 pp        │
├──────────────────────┼────────────────┼─────────────────────────┼─────────────────┤
│ w/o GCN              │ 43,599         │ 87.5%                   │ -4.6 pp         │
├──────────────────────┼────────────────┼─────────────────────────┼─────────────────┤
│